In [15]:
from groq import Groq
from dotenv import load_dotenv
from pydantic import BaseModel
from typing import Literal
from pypdf import PdfReader
from json import loads
from pathlib import Path
import os
import time

In [16]:
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise ValueError("No Api Key Found")

client = Groq(api_key=api_key)
model = "openai/gpt-oss-20b"

## Schema Design

This section defines the structured Pydantic models used throughout the notebook. These schemas keep the job description, parsed resume data, and final scoring output consistent across every LLM call.

In [17]:
class JobDescription(BaseModel):
    role: str
    required_skills: list[str]
    preferred_skills: list[str]
    minimum_experience: float | None
    education_requirements: list[str]
    responsibilities: list[str]

class Experience(BaseModel):
    company: str | None = None
    role: str | None = None
    duration: str | None = None
    description: str | None = None
    skills_used: list[str] = []

class Resume(BaseModel):
    name: str | None = None
    email: str | None = None
    phone: str | None = None
    total_experience_years: float | None = None
    skills: list[str] = []
    experiences: list[Experience] = []
    education: list[str] = []
    projects: list[str] = []
    certifications: list[str] = []

class Details(BaseModel):
    matching_skills: str | None = None
    missing_skills: str | None = None
    experience_match: Literal['Strong', 'Average', 'Weak'] = []
    recommendation: Literal['Shortlist', 'Waitlist', 'Reject'] = []
    short_verdict: str | None = None


class MatchResult(BaseModel):
    score: float
    details: Details



JobDescription_schema = JobDescription.model_json_schema()
resume_schema = Resume.model_json_schema()

## Job Description Prompting

The raw job posting is written here, along with the system and user prompts that ask the model to convert it into structured JSON matching the `JobDescription` schema.

In [18]:

JobDescriptionescription = """
Role: Software Engineer Trainee
Position: MERN Stack Developer
Date of Joining: Immediate
Employment Type: Permanent
Experience Required: Fresher / Entry Level
About the Role
We are looking for a highly motivated and enthusiastic MERN Stack Developer who is passionate about building modern web applications.

The ideal candidate should have a solid understanding of web fundamentals, a keen interest in learning new technologies, and the ability to work independently as well as in a team.

This is a great opportunity for freshers who want to kickstart their career in full-stack development using MongoDB, Express.js, React.js, and Node.js — and grow into a complete product developer.

Key Responsibilities
Design, Develop, and maintenance of full-stack web applications using the MERN stack (MongoDB, Express, React, Node.js).
Write clean, efficient, and well-structured code following standard development
Learn and apply concepts of RESTful APIs, state management, and asynchronous programming.
Collaborate with team members to understand requirements, propose solutions, and deliver results within timelines.
Participate in code reviews and continuously improve code
Troubleshoot issues, debug applications, and contribute to performance
Stay updated with emerging web technologies and best
Take ownership of assigned modules or features with minimal
Demonstrate curiosity and eagerness to learn new tools, frameworks, and concepts
Required Skills & Knowledge
Technical Skills (Basic to Intermediate Knowledge Expected):

Good understanding of JavaScript (ES6+), HTML5, and CSS3.
Basic familiarity with js — components, props, state, hooks.
Understanding of js and Express.js for backend development.
Knowledge of MongoDB — schema design, CRUD operations,
Understanding of REST APIs and JSON-based
Basic knowledge of Git for version
Familiarity with package managers (npm/yarn) and Postman/Thunder Client for API
Additional Advantage (Good to Have):
Exposure to TypeScript, NestJs, js, or Redux.
Awareness of deployment processes (e.g., Vercel, Render, or AWS/Azure basics).
Basic understanding of UI/UX design principles. Soft Skills & Learning Mindset
Strong communication skills able to explain ideas clearly and collaborate
Demonstrated self-learning capability  ability to research and learn new tools or frameworks independently.
Problem-solving mindset with an analytical and logical
Highly self-motivated with a passion for technology and
Positive attitude, team player, and willingness to take ownership of assigned
Open to feedback and eager to continuously
Educational Qualification
Bachelor's degree in computer science, Information Technology, or any equivalent
Candidates with personal or academic projects in MERN stack will be given"""


system_prompt = f"""
You are an expert HR assistant.

Your job is to analyze job descriptions and extract
structured information from them.

Return ONLY valid JSON matching this schema:

{JobDescription_schema}
IMPORTANT:
Do NOT return the schema itself.
Do NOT return fields like "properties", "title" or "type".
Fill the schema with actual information extracted from the job description.
For string or numeric fields with missing information, return null.
For list fields with no information, return [].
Never return null for a list field.
Do not invent information.
"""

user_prompt = f"""
Analyze the following job description:

{JobDescriptionescription}
"""

## Message Assembly

This cell prepares the chat payload sent to the model. Keeping the system and user messages separate makes the extraction prompt easier to inspect and adjust.

In [19]:
system_message = {
    "role": "system",
    "content": system_prompt
}

user_message = {
    "role": "user",
    "content": user_prompt
}

messages = [system_message, user_message]

## Job Extraction

The model is called here to transform the free-form job description into validated structured data. The JSON response is parsed and loaded into the `JobDescription` model before being reused later in the notebook.

In [20]:
response = client.chat.completions.create(
    model=model,
    messages=messages,
    response_format={"type": "json_object"},
)

data = loads(response.choices[0].message.content)
job = JobDescription(**data)
data

{'role': 'Software Engineer Trainee',
 'required_skills': ['JavaScript (ES6+)',
  'HTML5',
  'CSS3',
  'React.js (components, props, state, hooks)',
  'Node.js',
  'Express.js',
  'MongoDB (schema design, CRUD operations)',
  'RESTful APIs',
  'JSON',
  'Git',
  'npm/yarn',
  'Postman/Thunder Client'],
 'preferred_skills': ['TypeScript',
  'NestJs',
  'Redux',
  'Deployment (Vercel, Render, AWS, Azure)',
  'UI/UX design principles'],
 'minimum_experience': 0,
 'education_requirements': ["Bachelor's degree in Computer Science, Information Technology, or equivalent"],
 'responsibilities': ['Design, develop, and maintain full-stack web applications using the MERN stack',
  'Write clean, efficient, well-structured code following standard development practices',
  'Learn and apply concepts of RESTful APIs, state management, and asynchronous programming',
  'Collaborate with team members to understand requirements, propose solutions, and deliver results within timelines',
  'Participate in c

## Resume Parsing Utilities

These helper functions read PDF resumes, extract text, normalize the content into the `Resume` schema, and generate a final fit score by comparing the candidate against the selected job.

In [21]:
def read_resume(filepath):
    text = ""
    reader = PdfReader(filepath)
    for page in reader.pages:
        text += page.extract_text() or ""

    return text

def parse_resume(resume_text):
    system_prompt = f"""
    You are an expert resume parser.

    Extract information from the resume based on its meaning,
    not only based on exact section headings.

    Different resumes may use different headings.

    For example:
    - Experience
    - Professional Experience
    - Work History
    - Employment
    - Internships

    These may all contain relevant experience.

    Skills may also appear in the skills section, work experience,
    internships or projects.

    Return ONLY valid JSON matching this schema:

    {resume_schema}

    Important rules:

    1. Do not invent information.
    2. If a value is not available, return null.
    3. If a list has no information, return an empty list.
    4. Include internships inside experiences.
    5. Extract skills mentioned across the entire resume.
    """
    user_prompt = f"""
    Parse the following resume:

    {resume_text}
    """
    message_system={
        "role" : "system",
        "content" : system_prompt
    }
    message_user={
        "role" : "user",
        "content" : user_prompt
    }
    messages=[message_system, message_user]
    response_format={
        "type": "json_object"
    }

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        response_format=response_format
    )

    data = loads(response.choices[0].message.content)
    resume = Resume(**data)
    return resume

def final_score(job, resume):
    match_result = MatchResult.model_json_schema()
    prompt=f"""
    You're a senior HR Assitant, your job is to compare the job description and candidate resume
    and give your final verdict

    the job description being: {job}
    the resume being: {resume}

    You must strictly return Youre response in Valid JSON Format
    Format: {match_result}

    IMPORTANT:
    ->Do not invent anything
    ->Do not put the whole essay in details->verdict section, just how you feel candidate might be fit for institute and why
    ->Score should be strictly between 0-100
    """

    user_prompt = {
        "role": "user",
        "content": prompt
    }

    messages = [user_prompt]
    response_format = {
        "type": "json_object"
    }

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        response_format=response_format
    )

    data = loads(response.choices[0].message.content)
    final_response = MatchResult(**data)
    return final_response

## Batch Scoring

This final workflow loops through every PDF in `resumes/`, parses each candidate, scores the match against the structured job description, and stores a compact summary for review.

In [22]:
files = Path("resumes")
results = []

for resume_file in files.iterdir():
    if resume_file.suffix.lower() != ".pdf":
        continue
    raw_resume = read_resume(resume_file)
    parsed_resume = parse_resume(raw_resume)
    time.sleep(5)
    result = final_score(job, parsed_resume)
    time.sleep(5)
    results.append({
        "name": parsed_resume.name,
        "score": result.score,
        "details": result.details
    })



In [23]:
for result in results:
    print(result)

{'name': 'AARAV SHARMA', 'score': 82.0, 'details': Details(matching_skills='JavaScript, HTML, CSS, React.js, Node.js, Express.js, MongoDB, REST APIs, Git, Postman', missing_skills='JSON, npm/yarn', experience_match='Average', recommendation='Shortlist', short_verdict='Candidate meets all required technical skills, has a relevant CS degree, and is a solid fit for a Software Engineer Trainee role. Minor gaps in JSON usage and package manager familiarity, but overall strong suitability.')}
{'name': 'Priya Patel', 'score': 90.0, 'details': Details(matching_skills='JavaScript, HTML5, CSS3, React.js, Node.js, Express.js, MongoDB, RESTful APIs, JSON, Git, npm/yarn, Postman, TypeScript, Redux Toolkit, Vercel, Render', missing_skills='NestJs, UI/UX design principles, Azure', experience_match='Strong', recommendation='Shortlist', short_verdict='Strong fit due to full‑stack experience and matching core skills.')}
{'name': 'ROHAN MEHTA', 'score': 90.0, 'details': Details(matching_skills='["JavaScr